# Week 3 - KPI & Profitability Analysis

## Title

## Objective
Calculate financial KPIs, measure profitability, compare product and regional performance, and identify business insights from the final cleaned financial-retail dataset.

## Dataset

This notebook uses `financial_retail_clean.csv`, the validated Week 1 output. It includes 850 retail transactions spanning 2025 across revenue, cost, expense, profit, customer, product, region, city, and quantity fields.

## KPI Definitions

- Total Revenue = sum(Revenue)
- Total Expense = sum(Cost + Operating_Expense)
- Gross Profit = Revenue - Cost
- Net Profit = Gross Profit - Operating_Expense
- Profit Margin = Net Profit / Revenue * 100
- Cost Ratio = Total Expense / Revenue * 100
- Average Transaction Value = Revenue / Number of Transactions
- Product Contribution = Product Revenue / Total Revenue * 100
- Regional Contribution = Region Revenue / Total Revenue * 100

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
workspace_roots = [Path.cwd(), *Path.cwd().parents]
data_path = next(
    (
        root / 'data' / 'cleaned' / 'financial_retail_clean.csv'
        for root in workspace_roots
        if (root / 'data' / 'cleaned' / 'financial_retail_clean.csv').is_file()
    ),
    None,
)
if data_path is None:
    raise FileNotFoundError('Could not find data/cleaned/financial_retail_clean.csv.')
df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'])
df['Total_Expense'] = df['Cost'] + df['Operating_Expense']
df['Gross_Profit'] = df['Revenue'] - df['Cost']
df['Net_Profit'] = df['Gross_Profit'] - df['Operating_Expense']
df['Profit_Margin'] = df['Net_Profit'] / df['Revenue'] * 100
df['Cost_Ratio'] = df['Total_Expense'] / df['Revenue'] * 100
print('Total Revenue:', round(df['Revenue'].sum(), 2))
print('Total Expense:', round(df['Total_Expense'].sum(), 2))
print('Net Profit:', round(df['Net_Profit'].sum(), 2))
print('Profit Margin:', round(df['Net_Profit'].sum()/df['Revenue'].sum()*100, 2), '%')
print('Cost Ratio:', round(df['Total_Expense'].sum()/df['Revenue'].sum()*100, 2), '%')
print('Average Transaction Value:', round(df['Revenue'].mean(), 2))

Total Revenue: 61039689.51
Total Expense: 47121313.91
Net Profit: 13918375.6
Profit Margin: 22.8 %
Cost Ratio: 77.2 %
Average Transaction Value: 71811.4


## Overall Financial KPIs

The KPI summary provides the headline business performance for the full portfolio and serves as the anchor for downstream diagnostics.

In [2]:
kpi_summary = pd.DataFrame({
    'Metric':['Total Revenue','Total Expense','Gross Profit','Net Profit','Profit Margin %','Cost Ratio %','Average Transaction Value'],
    'Value':[df['Revenue'].sum(), df['Total_Expense'].sum(), df['Gross_Profit'].sum(), df['Net_Profit'].sum(), df['Net_Profit'].sum()/df['Revenue'].sum()*100, df['Total_Expense'].sum()/df['Revenue'].sum()*100, df['Revenue'].mean()]
})
print(kpi_summary.to_string(index=False))

                   Metric        Value
            Total Revenue 6.103969e+07
            Total Expense 4.712131e+07
             Gross Profit 2.160135e+07
               Net Profit 1.391838e+07
          Profit Margin % 2.280217e+01
             Cost Ratio % 7.719783e+01
Average Transaction Value 7.181140e+04


## Revenue Performance

Revenue performance is checked at a total, monthly, and regional level to identify where the business is expanding or contracting.

In [3]:
monthly_revenue = df.groupby(df['Date'].dt.to_period('M').astype(str))['Revenue'].sum().reset_index(name='Revenue')
print(monthly_revenue.round(2).to_string(index=False))
region_revenue = df.groupby('Region')['Revenue'].sum().sort_values(ascending=False)
print('Region revenue:\n', region_revenue.to_string())

   Date    Revenue
2025-01 4865781.43
2025-02 5508151.11
2025-03 4383197.83
2025-04 4982268.37
2025-05 5884249.97
2025-06 6296528.44
2025-07 5391616.52
2025-08 5092172.61
2025-09 4373313.75
2025-10 5762384.55
2025-11 4043125.98
2025-12 4456898.95
Region revenue:
 Region
West     34309749.58
South    13379888.38
North    10818323.38
East      2531728.17


## Expense Performance

Expense analysis tracks direct and operating costs to understand the cost structure behind revenue generation.

In [4]:
category_expense = df.groupby('Category')['Total_Expense'].sum().sort_values(ascending=False)
print(category_expense.to_string())
region_expense = df.groupby('Region')['Total_Expense'].sum().sort_values(ascending=False)
print('Region expense:\n', region_expense.to_string())

Category
Electronics    40615084.06
Furniture       4042514.13
Accessories     2463715.72
Region expense:
 Region
West     26385593.90
South    10323185.66
North     8447654.84
East      1964879.51


## Profitability Analysis

This section measures profit contribution by category and region and compares margin quality across segments.

In [5]:
category_profit = df.groupby('Category').agg(Revenue=('Revenue','sum'), Profit=('Net_Profit','sum')).assign(Margin=lambda x: x['Profit']/x['Revenue']*100).sort_values('Revenue', ascending=False)
print(category_profit.round(2).to_string())
region_profit = df.groupby('Region').agg(Revenue=('Revenue','sum'), Profit=('Net_Profit','sum')).assign(Margin=lambda x: x['Profit']/x['Revenue']*100).sort_values('Revenue', ascending=False)
print('Region profitability:\n', region_profit.round(2).to_string())

                 Revenue       Profit  Margin
Category                                     
Electronics  52680285.38  12065201.32   22.90
Furniture     5152662.82   1110148.69   21.55
Accessories   3206741.31    743025.59   23.17
Region profitability:
             Revenue      Profit  Margin
Region                                 
West    34309749.58  7924155.68   23.10
South   13379888.38  3056702.72   22.85
North   10818323.38  2370668.54   21.91
East     2531728.17   566848.66   22.39


## Profit Margin Analysis

Margin analysis determines whether revenue growth is being converted to sustainable profits.

In [6]:
profit_margin_by_category = df.groupby('Category').apply(lambda x: (x['Net_Profit'].sum() / x['Revenue'].sum() * 100)).sort_values(ascending=False)
print('Profit margin by category:', profit_margin_by_category.round(2).to_string())
profit_margin_by_region = df.groupby('Region').apply(lambda x: (x['Net_Profit'].sum() / x['Revenue'].sum() * 100)).sort_values(ascending=False)
print('Profit margin by region:\n', profit_margin_by_region.round(2).to_string())

Profit margin by category: Category
Accessories    23.17
Electronics    22.90
Furniture      21.55
Profit margin by region:
 Region
West     23.10
South    22.85
East     22.39
North    21.91


## Cost-to-Revenue Analysis

Cost-to-revenue analysis indicates how much of each sales rupee is absorbed by direct and operating costs.

In [7]:
cost_ratio_by_category = df.groupby('Category').apply(lambda x: (x['Total_Expense'].sum() / x['Revenue'].sum() * 100)).sort_values(ascending=False)
print('Cost ratio by category:', cost_ratio_by_category.round(2).to_string())
cost_ratio_by_region = df.groupby('Region').apply(lambda x: (x['Total_Expense'].sum() / x['Revenue'].sum() * 100)).sort_values(ascending=False)
print('Cost ratio by region:\n', cost_ratio_by_region.round(2).to_string())

Cost ratio by category: Category
Furniture      78.45
Electronics    77.10
Accessories    76.83
Cost ratio by region:
 Region
North    78.09
East     77.61
South    77.15
West     76.90


## Product Profitability

The highest sales products are not always the highest-profit products; margin quality matters.

In [8]:
product_profitability = df.groupby('Product').agg(Revenue=('Revenue','sum'), Expense=('Total_Expense','sum'), Profit=('Net_Profit','sum')).assign(Margin=lambda x: x['Profit']/x['Revenue']*100, Contribution=lambda x: x['Revenue']/df['Revenue'].sum()*100).sort_values('Revenue', ascending=False)
print(product_profitability.head(10).round(2).to_string())

                  Revenue      Expense      Profit  Margin  Contribution
Product                                                                 
Laptop        28023050.88  21606473.38  6416577.50   22.90         45.91
Smartphone    13624053.70  10622140.06  3001913.64   22.03         22.32
Tablet         4273728.49   3256504.33  1017224.16   23.80          7.00
Monitor        3947910.29   3050455.58   897454.71   22.73          6.47
Desk           3185392.74   2525967.47   659425.27   20.70          5.22
Printer        2232128.32   1694456.88   537671.44   24.09          3.66
Office Chair   1967270.08   1516546.66   450723.42   22.91          3.22
Smartwatch     1279243.67    966093.61   313150.06   24.48          2.10
Headphones     1174588.23    921716.95   252871.28   21.53          1.92
Unknown         603622.26    402052.23   201570.03   33.39          0.99


## Regional Profitability

Regional performance should be evaluated both by absolute contribution and by margin quality.

In [9]:
regional_profitability = df.groupby('Region').agg(Revenue=('Revenue','sum'), Expense=('Total_Expense','sum'), Profit=('Net_Profit','sum')).assign(Margin=lambda x: x['Profit']/x['Revenue']*100, Contribution=lambda x: x['Revenue']/df['Revenue'].sum()*100).sort_values('Revenue', ascending=False)
print(regional_profitability.round(2).to_string())

            Revenue      Expense      Profit  Margin  Contribution
Region                                                            
West    34309749.58  26385593.90  7924155.68   23.10         56.21
South   13379888.38  10323185.66  3056702.72   22.85         21.92
North   10818323.38   8447654.84  2370668.54   21.91         17.72
East     2531728.17   1964879.51   566848.66   22.39          4.15


## Monthly Growth

Revenue, expense, and profit growth are measured month-over-month to identify expansion and contraction periods.

In [10]:
monthly_kpi = df.groupby(df['Date'].dt.to_period('M').astype(str)).agg(Revenue=('Revenue','sum'), Expense=('Total_Expense','sum'), Profit=('Net_Profit','sum')).reset_index()
monthly_kpi['Revenue_Growth_pct'] = monthly_kpi['Revenue'].pct_change().mul(100)
monthly_kpi['Profit_Growth_pct'] = monthly_kpi['Profit'].pct_change().mul(100)
monthly_kpi['Cost_Growth_pct'] = monthly_kpi['Expense'].pct_change().mul(100)
print(monthly_kpi.round(2).to_string(index=False))

   Date    Revenue    Expense     Profit  Revenue_Growth_pct  Profit_Growth_pct  Cost_Growth_pct
2025-01 4865781.43 3795915.11 1069866.32                 NaN                NaN              NaN
2025-02 5508151.11 4206685.38 1301465.73               13.20              21.65            10.82
2025-03 4383197.83 3434314.93  948882.90              -20.42             -27.09           -18.36
2025-04 4982268.37 3820061.95 1162206.42               13.67              22.48            11.23
2025-05 5884249.97 4593214.91 1291035.06               18.10              11.08            20.24
2025-06 6296528.44 4866794.17 1429734.27                7.01              10.74             5.96
2025-07 5391616.52 4086486.28 1305130.24              -14.37              -8.72           -16.03
2025-08 5092172.61 3975075.37 1117097.24               -5.55             -14.41            -2.73
2025-09 4373313.75 3255312.67 1118001.08              -14.12               0.08           -18.11
2025-10 5762384.55 4595948.55 

## High-Performing Areas

The highest-performing parts of the portfolio are the segments with the strongest revenue and profit contribution.

In [11]:
top_regions = df.groupby('Region')['Net_Profit'].sum().sort_values(ascending=False)
print('Top regions by profit:', top_regions.to_string())
top_products = df.groupby('Product')['Net_Profit'].sum().sort_values(ascending=False)
print('Top products by profit:\n', top_products.head(10).to_string())

Top regions by profit: Region
West     7924155.68
South    3056702.72
North    2370668.54
East      566848.66
Top products by profit:
 Product
Laptop          6416577.50
Smartphone      3001913.64
Tablet          1017224.16
Monitor          897454.71
Desk             659425.27
Printer          537671.44
Office Chair     450723.42
Smartwatch       313150.06
Headphones       252871.28
Unknown          201570.03


## Low-Performing Areas

Lower-performing segments are tracked by absolute profit and margin to separate weak volume areas from weak efficiency areas.

In [12]:
low_margin_regions = df.groupby('Region').apply(lambda x: (x['Net_Profit'].sum() / x['Revenue'].sum() * 100)).sort_values()
print('Lowest margin regions:', low_margin_regions.to_string())
low_margin_categories = df.groupby('Category').apply(lambda x: (x['Net_Profit'].sum() / x['Revenue'].sum() * 100)).sort_values()
print('Lowest margin categories:\n', low_margin_categories.to_string())

Lowest margin regions: Region
North    21.913456
East     22.389792
South    22.845502
West     23.095930
Lowest margin categories:
 Category
Furniture      21.545145
Electronics    22.902688
Accessories    23.170737


## Financial Anomaly Detection

This section uses IQR thresholds to highlight unusually high- or low-performing transactions or margins. These are flagged as potential anomalies and require review rather than automatic action.

In [13]:
revenue_q1, revenue_q3 = df['Revenue'].quantile([0.25, 0.75])
revenue_iqr = revenue_q3 - revenue_q1
revenue_upper = revenue_q3 + 1.5 * revenue_iqr
margin_q1, margin_q3 = df['Profit_Margin'].quantile([0.25, 0.75])
margin_iqr = margin_q3 - margin_q1
margin_lower = margin_q1 - 1.5 * margin_iqr
flagged = df[(df['Revenue'] > revenue_upper) | (df['Profit_Margin'] < margin_lower)]
print('Revenue upper IQR bound:', round(revenue_upper, 2))
print('Margin lower IQR bound:', round(margin_lower, 2))
print('Flagged anomalies:', flagged.shape[0])
if not flagged.empty: print(flagged[['Transaction_ID','Revenue','Net_Profit','Profit_Margin']].head(10).to_string(index=False))

Revenue upper IQR bound: 207029.28
Margin lower IQR bound: -2.94
Flagged anomalies: 55
Transaction_ID   Revenue  Net_Profit  Profit_Margin
     TXN100614 322129.56    58172.23      18.058644
     TXN100688 386250.57    89398.47      23.145201
     TXN100270 273666.48    73561.52      26.879989
     TXN100682 505474.41   111733.45      22.104670
     TXN100026 214649.24    60484.98      28.178520
     TXN100552 290605.91    95305.40      32.795410
     TXN100338 359161.29   124074.91      34.545736
     TXN100373 233569.85    20960.06       8.973787
     TXN100329 315571.96    77433.69      24.537570
     TXN100817 349019.17   121639.94      34.851937


## KPI Visualizations

The KPI visualizations support the decision-making narrative by translating metrics into business-ready summaries.

## Business Insights

1. Electronics drives the underlying profit model.
2. West region delivers the strongest performance.
3. Accessories have the best margin efficiency.
4. Laptop is the primary value driver.
5. Seasonal peaks are concentrated in May and June.
6. Cost ratios are significant and require active management.
7. Large ticket sales remain important.
8. Anomaly review is valuable for data-quality and forecast stability.

## Recommendations

- Protect pricing and stock for Electronics and Laptop.
- Increase investment in the West region and optimize promotions there.
- Use accessory bundling to raise margin quality.
- Align inventory and staffing with seasonal demand peaks.
- Review unusual transactions before using them for future planning.

## Conclusion

The KPI framework shows a healthy and profitable portfolio with strong electronics dominance, regional concentration, and seasonal revenue variability. Margin focus, inventory planning, and anomaly review are the most important next steps for business improvement.